# Notebook 6 — Model Training and Evaluation

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import joblib
import os
import json

## 2. Loading Processed Data

In [2]:
X_train_processed = np.load("../data/artifacts/X_train_processed.npy")
X_validation_processed = np.load("../data/artifacts/X_validation_processed.npy")
X_test_processed = np.load("../data/artifacts/X_test_processed.npy")

y_train = np.load("../data/artifacts/y_train.npy")
y_validation = np.load("../data/artifacts/y_validation.npy")
y_test = np.load("../data/artifacts/y_test.npy")

print("X_train:", X_train_processed.shape)
print("X_validation:", X_validation_processed.shape)
print("X_test:", X_test_processed.shape)

print("y_train:", y_train.shape)
print("y_validation:", y_validation.shape)
print("y_test:", y_test.shape)

X_train: (67533, 41)
X_validation: (14471, 41)
X_test: (14472, 41)
y_train: (67533,)
y_validation: (14471,)
y_test: (14472,)


## 3. Checking Target Distribution

In [3]:
print("Training target distribution:")
print(pd.Series(y_train).value_counts())

print("\nTraining target percentages:")
print(
    pd.Series(y_train)
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nValidation target distribution:")
print(pd.Series(y_validation).value_counts())

print("\nTest target distribution:")
print(pd.Series(y_test).value_counts())

Training target distribution:
0    61436
1     6097
Name: count, dtype: int64

Training target percentages:
0    90.97
1     9.03
Name: proportion, dtype: float64

Validation target distribution:
0    13698
1      773
Name: count, dtype: int64

Test target distribution:
0    13515
1      957
Name: count, dtype: int64


## 4. Baseline Model

In [4]:
# Majority-class baseline
baseline_predictions = np.zeros(len(y_validation), dtype=int)

baseline_accuracy = accuracy_score(y_validation, baseline_predictions)
baseline_precision = precision_score(
    y_validation,
    baseline_predictions,
    zero_division=0
)
baseline_recall = recall_score(
    y_validation,
    baseline_predictions,
    zero_division=0
)
baseline_f1 = f1_score(
    y_validation,
    baseline_predictions,
    zero_division=0
)

print("Baseline Model Performance")
print("--------------------------")
print(f"Accuracy:  {baseline_accuracy:.4f}")
print(f"Precision: {baseline_precision:.4f}")
print(f"Recall:    {baseline_recall:.4f}")
print(f"F1-score:  {baseline_f1:.4f}")

Baseline Model Performance
--------------------------
Accuracy:  0.9466
Precision: 0.0000
Recall:    0.0000
F1-score:  0.0000


## 5. Logistic Regression

In [5]:
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(
    X_train_processed,
    y_train
)

logistic_predictions = logistic_model.predict(
    X_validation_processed
)

logistic_probabilities = logistic_model.predict_proba(
    X_validation_processed
)[:, 1]

In [6]:
logistic_accuracy = accuracy_score(
    y_validation,
    logistic_predictions
)

logistic_precision = precision_score(
    y_validation,
    logistic_predictions,
    zero_division=0
)

logistic_recall = recall_score(
    y_validation,
    logistic_predictions,
    zero_division=0
)

logistic_f1 = f1_score(
    y_validation,
    logistic_predictions,
    zero_division=0
)

logistic_roc_auc = roc_auc_score(
    y_validation,
    logistic_probabilities
)

print("Logistic Regression Performance")
print("--------------------------------")
print(f"Accuracy:  {logistic_accuracy:.4f}")
print(f"Precision: {logistic_precision:.4f}")
print(f"Recall:    {logistic_recall:.4f}")
print(f"F1-score:  {logistic_f1:.4f}")
print(f"ROC-AUC:   {logistic_roc_auc:.4f}")

Logistic Regression Performance
--------------------------------
Accuracy:  0.9436
Precision: 0.0612
Recall:    0.0039
F1-score:  0.0073
ROC-AUC:   0.6053


### 5.1 Improved Logistic Regression

In [7]:
improved_logistic_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    C=1.0,
    solver="liblinear",
    random_state=42
)

improved_logistic_model.fit(
    X_train_processed,
    y_train
)

improved_logistic_probabilities = improved_logistic_model.predict_proba(
    X_validation_processed
)[:, 1]

improved_logistic_predictions = (
    improved_logistic_probabilities >= 0.50
).astype(int)

print("Improved Logistic Regression - Validation Results")
print("-------------------------------------------------")
print(
    f"Accuracy : {accuracy_score(y_validation, improved_logistic_predictions):.4f}"
)
print(
    f"Precision: {precision_score(y_validation, improved_logistic_predictions, zero_division=0):.4f}"
)
print(
    f"Recall   : {recall_score(y_validation, improved_logistic_predictions, zero_division=0):.4f}"
)
print(
    f"F1-Score : {f1_score(y_validation, improved_logistic_predictions, zero_division=0):.4f}"
)
print(
    f"ROC-AUC  : {roc_auc_score(y_validation, improved_logistic_probabilities):.4f}"
)

Improved Logistic Regression - Validation Results
-------------------------------------------------
Accuracy : 0.0804
Precision: 0.0546
Recall   : 0.9948
F1-Score : 0.1036
ROC-AUC  : 0.6066


### 5.2 Threshold Tuning for Improved Logistic Regression

In [8]:
logistic_threshold_results = []

for threshold in np.arange(0.05, 0.96, 0.01):

    predictions = (
        improved_logistic_probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_validation,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_validation,
        predictions,
        zero_division=0
    )

    logistic_threshold_results.append({
        "Threshold": round(threshold, 2),
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

logistic_threshold_results_df = pd.DataFrame(
    logistic_threshold_results
)

logistic_threshold_results_df.sort_values(
    by="F1",
    ascending=False
).head(10)

,Threshold,Precision,Recall,F1
69,0.74,0.090036,0.448900,0.149989
68,0.73,0.088778,0.460543,0.148861
67,0.72,0.088102,0.476067,0.148687
66,0.71,0.087269,0.487710,0.148046
70,0.75,0.088828,0.421734,0.146748
65,0.70,0.085689,0.498060,0.146221
71,0.76,0.089157,0.401035,0.145882
64,0.69,0.083832,0.507115,0.143880
63,0.68,0.082166,0.516171,0.141766
72,0.77,0.086755,0.363519,0.140080


## 6. Decision Tree

In [9]:
decision_tree_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=8
)

decision_tree_model.fit(
    X_train_processed,
    y_train
)

decision_tree_predictions = decision_tree_model.predict(
    X_validation_processed
)

decision_tree_probabilities = decision_tree_model.predict_proba(
    X_validation_processed
)[:, 1]

In [10]:
decision_tree_accuracy = accuracy_score(
    y_validation,
    decision_tree_predictions
)

decision_tree_precision = precision_score(
    y_validation,
    decision_tree_predictions,
    zero_division=0
)

decision_tree_recall = recall_score(
    y_validation,
    decision_tree_predictions,
    zero_division=0
)

decision_tree_f1 = f1_score(
    y_validation,
    decision_tree_predictions,
    zero_division=0
)

decision_tree_roc_auc = roc_auc_score(
    y_validation,
    decision_tree_probabilities
)

print("Decision Tree Performance")
print("-------------------------")
print(f"Accuracy:  {decision_tree_accuracy:.4f}")
print(f"Precision: {decision_tree_precision:.4f}")
print(f"Recall:    {decision_tree_recall:.4f}")
print(f"F1-score:  {decision_tree_f1:.4f}")
print(f"ROC-AUC:   {decision_tree_roc_auc:.4f}")

Decision Tree Performance
-------------------------
Accuracy:  0.9406
Precision: 0.1293
Recall:    0.0194
F1-score:  0.0337
ROC-AUC:   0.5749


## 7. Random Forest

In [11]:
random_forest_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(
    X_train_processed,
    y_train
)

random_forest_predictions = random_forest_model.predict(
    X_validation_processed
)

random_forest_probabilities = random_forest_model.predict_proba(
    X_validation_processed
)[:, 1]

In [12]:
random_forest_accuracy = accuracy_score(
    y_validation,
    random_forest_predictions
)

random_forest_precision = precision_score(
    y_validation,
    random_forest_predictions,
    zero_division=0
)

random_forest_recall = recall_score(
    y_validation,
    random_forest_predictions,
    zero_division=0
)

random_forest_f1 = f1_score(
    y_validation,
    random_forest_predictions,
    zero_division=0
)

random_forest_roc_auc = roc_auc_score(
    y_validation,
    random_forest_probabilities
)

print("Random Forest Performance")
print("-------------------------")
print(f"Accuracy:  {random_forest_accuracy:.4f}")
print(f"Precision: {random_forest_precision:.4f}")
print(f"Recall:    {random_forest_recall:.4f}")
print(f"F1-score:  {random_forest_f1:.4f}")
print(f"ROC-AUC:   {random_forest_roc_auc:.4f}")

Random Forest Performance
-------------------------
Accuracy:  0.9466
Precision: 0.0000
Recall:    0.0000
F1-score:  0.0000
ROC-AUC:   0.6458


## 8. Initial Model Comparison

In [13]:
results = pd.DataFrame({
    "Model": [
        "Baseline",
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "Accuracy": [
        baseline_accuracy,
        logistic_accuracy,
        decision_tree_accuracy,
        random_forest_accuracy
    ],
    "Precision": [
        baseline_precision,
        logistic_precision,
        decision_tree_precision,
        random_forest_precision
    ],
    "Recall": [
        baseline_recall,
        logistic_recall,
        decision_tree_recall,
        random_forest_recall
    ],
    "F1-score": [
        baseline_f1,
        logistic_f1,
        decision_tree_f1,
        random_forest_f1
    ],
    "ROC-AUC": [
        np.nan,
        logistic_roc_auc,
        decision_tree_roc_auc,
        random_forest_roc_auc
    ]
})

results

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Baseline,0.946583,0.000000,0.000000,0.000000,NaN
1,Logistic Regression,0.943611,0.061224,0.003881,0.007299,0.605273
2,Decision Tree,0.940640,0.129310,0.019405,0.033746,0.574906
3,Random Forest,0.946583,0.000000,0.000000,0.000000,0.645833


## 9. Threshold Analysis

In [14]:
thresholds = [0.50, 0.40, 0.30, 0.20, 0.10, 0.05]

threshold_results = []

for threshold in thresholds:
    predictions = (
        random_forest_probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_validation,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_validation,
        predictions,
        zero_division=0
    )

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df

,Threshold,Precision,Recall,F1-score
0,0.50,0.000000,0.000000,0.000000
1,0.40,0.400000,0.002587,0.005141
2,0.30,0.159574,0.019405,0.034602
3,0.20,0.142185,0.195343,0.164578
4,0.10,0.094256,0.467012,0.156854
5,0.05,0.061262,0.896507,0.114688


In [15]:
best_threshold_row = threshold_df.loc[
    threshold_df["F1-score"].idxmax()
]

print("Best threshold based on F1-score:")
print(best_threshold_row)

Best threshold based on F1-score:
Threshold    0.200000
Precision    0.142185
Recall       0.195343
F1-score     0.164578
Name: 3, dtype: float64


## 10. Handling Class Imbalance

In [16]:
balanced_rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

balanced_rf_model.fit(
    X_train_processed,
    y_train
)

balanced_rf_predictions = balanced_rf_model.predict(
    X_validation_processed
)

balanced_rf_probabilities = balanced_rf_model.predict_proba(
    X_validation_processed
)[:, 1]

In [17]:
balanced_rf_accuracy = accuracy_score(
    y_validation,
    balanced_rf_predictions
)

balanced_rf_precision = precision_score(
    y_validation,
    balanced_rf_predictions,
    zero_division=0
)

balanced_rf_recall = recall_score(
    y_validation,
    balanced_rf_predictions,
    zero_division=0
)

balanced_rf_f1 = f1_score(
    y_validation,
    balanced_rf_predictions,
    zero_division=0
)

balanced_rf_roc_auc = roc_auc_score(
    y_validation,
    balanced_rf_probabilities
)

print("Balanced Random Forest Performance")
print("----------------------------------")
print(f"Accuracy:  {balanced_rf_accuracy:.4f}")
print(f"Precision: {balanced_rf_precision:.4f}")
print(f"Recall:    {balanced_rf_recall:.4f}")
print(f"F1-score:  {balanced_rf_f1:.4f}")
print(f"ROC-AUC:   {balanced_rf_roc_auc:.4f}")

Balanced Random Forest Performance
----------------------------------
Accuracy:  0.8096
Precision: 0.1141
Recall:    0.3790
F1-score:  0.1753
ROC-AUC:   0.6420


## 11. Threshold Tuning for Balanced Random Forest

In [18]:
thresholds = [0.50, 0.40, 0.30, 0.20, 0.10, 0.05]

balanced_threshold_results = []

for threshold in thresholds:
    predictions = (
        balanced_rf_probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_validation,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_validation,
        predictions,
        zero_division=0
    )

    balanced_threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    })

balanced_threshold_df = pd.DataFrame(
    balanced_threshold_results
)

balanced_threshold_df

,Threshold,Precision,Recall,F1-score
0,0.50,0.114052,0.379043,0.175344
1,0.40,0.086829,0.498060,0.147878
2,0.30,0.066183,0.720569,0.121232
3,0.20,0.056663,0.974127,0.107097
4,0.10,0.053550,1.000000,0.101657
5,0.05,0.053417,1.000000,0.101417


In [19]:
best_balanced_threshold_row = balanced_threshold_df.loc[
    balanced_threshold_df["F1-score"].idxmax()
]

print("Best threshold for Balanced Random Forest:")
print(best_balanced_threshold_row)

Best threshold for Balanced Random Forest:
Threshold    0.500000
Precision    0.114052
Recall       0.379043
F1-score     0.175344
Name: 0, dtype: float64


## 12. Hyperparameter Tuning

In [20]:
param_distributions = {
    "n_estimators": [100, 200, 300],
    "max_depth": [8, 12, 16, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "class_weight": ["balanced", "balanced_subsample"]
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ),
    param_distributions=param_distributions,
    n_iter=15,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(
    X_train_processed,
    y_train
)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'class_weight': ['balanced', 'balanced_subsample'], 'max_depth': [8, 12, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",15
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a fu

In [21]:
print("Best Parameters:")
print(random_search.best_params_)

print("\nBest Cross-Validation F1-score:")
print(f"{random_search.best_score_:.4f}")

Best Parameters:
{'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_depth': 8, 'class_weight': 'balanced_subsample'}

Best Cross-Validation F1-score:
0.1344


## 13. Tuned Random Forest Evaluation

In [22]:
tuned_rf_model = random_search.best_estimator_

tuned_rf_predictions = tuned_rf_model.predict(
    X_validation_processed
)

tuned_rf_probabilities = tuned_rf_model.predict_proba(
    X_validation_processed
)[:, 1]

In [23]:
tuned_rf_accuracy = accuracy_score(
    y_validation,
    tuned_rf_predictions
)

tuned_rf_precision = precision_score(
    y_validation,
    tuned_rf_predictions,
    zero_division=0
)

tuned_rf_recall = recall_score(
    y_validation,
    tuned_rf_predictions,
    zero_division=0
)

tuned_rf_f1 = f1_score(
    y_validation,
    tuned_rf_predictions,
    zero_division=0
)

tuned_rf_roc_auc = roc_auc_score(
    y_validation,
    tuned_rf_probabilities
)

print("Tuned Random Forest Performance")
print("--------------------------------")
print(f"Accuracy:  {tuned_rf_accuracy:.4f}")
print(f"Precision: {tuned_rf_precision:.4f}")
print(f"Recall:    {tuned_rf_recall:.4f}")
print(f"F1-score:  {tuned_rf_f1:.4f}")
print(f"ROC-AUC:   {tuned_rf_roc_auc:.4f}")

Tuned Random Forest Performance
--------------------------------
Accuracy:  0.7632
Precision: 0.0964
Recall:    0.4101
F1-score:  0.1561
ROC-AUC:   0.6344


## 14. Threshold Tuning for Tuned Random Forest

In [24]:
thresholds = [0.50, 0.40, 0.30, 0.20, 0.10, 0.05]

tuned_threshold_results = []

for threshold in thresholds:
    predictions = (
        tuned_rf_probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_validation,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_validation,
        predictions,
        zero_division=0
    )

    tuned_threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    })

tuned_threshold_df = pd.DataFrame(
    tuned_threshold_results
)

tuned_threshold_df

,Threshold,Precision,Recall,F1-score
0,0.50,0.096411,0.410091,0.156119
1,0.40,0.070957,0.667529,0.128278
2,0.30,0.057533,0.943079,0.108450
3,0.20,0.053614,1.000000,0.101771
4,0.10,0.053417,1.000000,0.101417
5,0.05,0.053417,1.000000,0.101417


In [25]:
best_tuned_threshold_row = tuned_threshold_df.loc[
    tuned_threshold_df["F1-score"].idxmax()
]

print("Best threshold for Tuned Random Forest:")
print(best_tuned_threshold_row)

Best threshold for Tuned Random Forest:
Threshold    0.500000
Precision    0.096411
Recall       0.410091
F1-score     0.156119
Name: 0, dtype: float64


## 15. Class Weight Tuning

In [26]:
class_weight_configs = {
    "weight_2": {0: 1, 1: 2},
    "weight_3": {0: 1, 1: 3},
    "weight_4": {0: 1, 1: 4},
    "weight_5": {0: 1, 1: 5},
    "weight_7": {0: 1, 1: 7},
    "weight_9": {0: 1, 1: 9},
    "balanced": "balanced"
}

class_weight_results = []
class_weight_models = {}

for name, weight in class_weight_configs.items():

    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        class_weight=weight,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_processed, y_train)

    probabilities = model.predict_proba(
        X_validation_processed
    )[:, 1]

    predictions = (probabilities >= 0.50).astype(int)

    accuracy = accuracy_score(y_validation, predictions)
    precision = precision_score(y_validation, predictions, zero_division=0)
    recall = recall_score(y_validation, predictions, zero_division=0)
    f1 = f1_score(y_validation, predictions, zero_division=0)
    roc_auc = roc_auc_score(y_validation, probabilities)

    class_weight_results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc
    })

    class_weight_models[name] = model


class_weight_results_df = pd.DataFrame(class_weight_results)

class_weight_results_df.sort_values(
    by="F1",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
4,weight_7,0.870914,0.143322,0.284605,0.190641,0.639089
6,balanced,0.809550,0.114052,0.379043,0.175344,0.642030
5,weight_9,0.809896,0.113067,0.373868,0.173626,0.641915
3,weight_5,0.914933,0.148773,0.125485,0.136140,0.646230
2,weight_4,0.934835,0.180451,0.062096,0.092397,0.646969
1,weight_3,0.943819,0.243590,0.024580,0.044653,0.644116
0,weight_2,0.946030,0.214286,0.003881,0.007624,0.649570


## 16. Fine Threshold Tuning for the Best Class Weight Model

In [27]:
best_weight_model = class_weight_models["weight_7"]

best_weight_probabilities = best_weight_model.predict_proba(
    X_validation_processed
)[:, 1]

threshold_results = []

for threshold in np.arange(0.05, 0.96, 0.01):

    predictions = (
        best_weight_probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_validation,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_validation,
        predictions,
        zero_division=0
    )

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

threshold_results_df = pd.DataFrame(threshold_results)

threshold_results_df.sort_values(
    by="F1",
    ascending=False
).head(10)

,Threshold,Precision,Recall,F1
46,0.51,0.149091,0.265201,0.190875
45,0.50,0.143322,0.284605,0.190641
44,0.49,0.136848,0.298836,0.187729
43,0.48,0.132675,0.313066,0.186369
47,0.52,0.148953,0.239327,0.183623
42,0.47,0.127931,0.324709,0.183547
48,0.53,0.154270,0.217335,0.180451
40,0.45,0.121076,0.349288,0.179820
39,0.44,0.118766,0.363519,0.179038
41,0.46,0.122090,0.332471,0.178596


## 17. Final Model Selection and Validation

In [28]:
final_model = class_weight_models["weight_7"]

final_threshold = 0.51

final_validation_probabilities = final_model.predict_proba(
    X_validation_processed
)[:, 1]

final_validation_predictions = (
    final_validation_probabilities >= final_threshold
).astype(int)

final_validation_accuracy = accuracy_score(
    y_validation,
    final_validation_predictions
)

final_validation_precision = precision_score(
    y_validation,
    final_validation_predictions,
    zero_division=0
)

final_validation_recall = recall_score(
    y_validation,
    final_validation_predictions,
    zero_division=0
)

final_validation_f1 = f1_score(
    y_validation,
    final_validation_predictions,
    zero_division=0
)

final_validation_roc_auc = roc_auc_score(
    y_validation,
    final_validation_probabilities
)

print("FINAL MODEL CONFIGURATION")
print("-------------------------")

print("Model: Random Forest")
print("Class Weight: {0: 1, 1: 7}")
print(f"Threshold: {final_threshold:.2f}")

print("\nFinal Validation Results")
print("------------------------")

print(f"Accuracy : {final_validation_accuracy:.4f}")
print(f"Precision: {final_validation_precision:.4f}")
print(f"Recall   : {final_validation_recall:.4f}")
print(f"F1-Score : {final_validation_f1:.4f}")
print(f"ROC-AUC  : {final_validation_roc_auc:.4f}")

FINAL MODEL CONFIGURATION
-------------------------
Model: Random Forest
Class Weight: {0: 1, 1: 7}
Threshold: 0.51

Final Validation Results
------------------------
Accuracy : 0.8799
Precision: 0.1491
Recall   : 0.2652
F1-Score : 0.1909
ROC-AUC  : 0.6391


## 18. Final Validation Confusion Matrix

In [29]:
validation_cm = confusion_matrix(
    y_validation,
    final_validation_predictions
)

print("Validation Confusion Matrix:")
print(validation_cm)

tn, fp, fn, tp = validation_cm.ravel()

print("\nTrue Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

Validation Confusion Matrix:
[[12528  1170]
 [  568   205]]

True Negatives : 12528
False Positives: 1170
False Negatives: 568
True Positives : 205


## 19. Final Validation Classification Report

In [30]:
validation_report = classification_report(
    y_validation,
    final_validation_predictions,
    target_names=["On-time", "Late"],
    zero_division=0
)

print(validation_report)

              precision    recall  f1-score   support

     On-time       0.96      0.91      0.94     13698
        Late       0.15      0.27      0.19       773

    accuracy                           0.88     14471
   macro avg       0.55      0.59      0.56     14471
weighted avg       0.91      0.88      0.90     14471



## 20. Final Test Evaluation

In [31]:
final_test_probabilities = final_model.predict_proba(
    X_test_processed
)[:, 1]

final_test_predictions = (
    final_test_probabilities >= final_threshold
).astype(int)

final_test_accuracy = accuracy_score(
    y_test,
    final_test_predictions
)

final_test_precision = precision_score(
    y_test,
    final_test_predictions,
    zero_division=0
)

final_test_recall = recall_score(
    y_test,
    final_test_predictions,
    zero_division=0
)

final_test_f1 = f1_score(
    y_test,
    final_test_predictions,
    zero_division=0
)

final_test_roc_auc = roc_auc_score(
    y_test,
    final_test_probabilities
)

print("FINAL TEST PERFORMANCE")
print("----------------------")

print(f"Accuracy : {final_test_accuracy:.4f}")
print(f"Precision: {final_test_precision:.4f}")
print(f"Recall   : {final_test_recall:.4f}")
print(f"F1-Score : {final_test_f1:.4f}")
print(f"ROC-AUC  : {final_test_roc_auc:.4f}")
print(f"Threshold: {final_threshold:.2f}")

FINAL TEST PERFORMANCE
----------------------
Accuracy : 0.8502
Precision: 0.0498
Recall   : 0.0700
F1-Score : 0.0582
ROC-AUC  : 0.4037
Threshold: 0.51


## 21. Final Test Confusion Matrix

In [32]:
test_cm = confusion_matrix(
    y_test,
    final_test_predictions
)

print("Final Test Confusion Matrix:")
print(test_cm)

tn, fp, fn, tp = test_cm.ravel()

print("\nTrue Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

Final Test Confusion Matrix:
[[12237  1278]
 [  890    67]]

True Negatives : 12237
False Positives: 1278
False Negatives: 890
True Positives : 67


## 22. Final Test Classification Report

In [33]:
test_report = classification_report(
    y_test,
    final_test_predictions,
    target_names=["On-time", "Late"],
    zero_division=0
)

print(test_report)

              precision    recall  f1-score   support

     On-time       0.93      0.91      0.92     13515
        Late       0.05      0.07      0.06       957

    accuracy                           0.85     14472
   macro avg       0.49      0.49      0.49     14472
weighted avg       0.87      0.85      0.86     14472



## 23. Saving the Final Model and Results

In [34]:
artifact_dir = "../data/artifacts"

os.makedirs(artifact_dir, exist_ok=True)

# Save final model
joblib.dump(
    final_model,
    os.path.join(artifact_dir, "final_model.joblib")
)

# Save final results
final_results = pd.DataFrame([{
    "Model": "Random Forest",
    "Class Weight": "{0:1, 1:7}",
    "Threshold": final_threshold,

    "Validation Accuracy": final_validation_accuracy,
    "Validation Precision": final_validation_precision,
    "Validation Recall": final_validation_recall,
    "Validation F1": final_validation_f1,
    "Validation ROC-AUC": final_validation_roc_auc,

    "Test Accuracy": final_test_accuracy,
    "Test Precision": final_test_precision,
    "Test Recall": final_test_recall,
    "Test F1": final_test_f1,
    "Test ROC-AUC": final_test_roc_auc
}])

final_results.to_csv(
    os.path.join(artifact_dir, "final_results.csv"),
    index=False
)

# Save validation confusion matrix
pd.DataFrame(
    validation_cm,
    index=["Actual On-time", "Actual Late"],
    columns=["Predicted On-time", "Predicted Late"]
).to_csv(
    os.path.join(
        artifact_dir,
        "validation_confusion_matrix.csv"
    )
)

# Save test confusion matrix
pd.DataFrame(
    test_cm,
    index=["Actual On-time", "Actual Late"],
    columns=["Predicted On-time", "Predicted Late"]
).to_csv(
    os.path.join(
        artifact_dir,
        "test_confusion_matrix.csv"
    )
)

# Save validation classification report
validation_report_dict = classification_report(
    y_validation,
    final_validation_predictions,
    target_names=["On-time", "Late"],
    output_dict=True,
    zero_division=0
)

pd.DataFrame(validation_report_dict).transpose().to_csv(
    os.path.join(
        artifact_dir,
        "validation_classification_report.csv"
    )
)

# Save test classification report
test_report_dict = classification_report(
    y_test,
    final_test_predictions,
    target_names=["On-time", "Late"],
    output_dict=True,
    zero_division=0
)

pd.DataFrame(test_report_dict).transpose().to_csv(
    os.path.join(
        artifact_dir,
        "test_classification_report.csv"
    )
)

# Save model metadata
metadata = {
    "model": "RandomForestClassifier",
    "class_weight": {
        "0": 1,
        "1": 7
    },
    "threshold": final_threshold,
    "validation_f1": final_validation_f1,
    "test_f1": final_test_f1
}

with open(
    os.path.join(artifact_dir, "final_model_metadata.json"),
    "w"
) as f:
    json.dump(metadata, f, indent=4)

print("Final artifacts saved successfully.")

Final artifacts saved successfully.
